In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import numpy as np
import pandas as pd
import shutil

FRAME_LENGTH = 40960

CLEAN_SOURCES = {
    TRAIN: get_unet_path(STAGE_AUGMENTED, TRAIN, CommSignal2),
    VAL:   get_unet_path(STAGE_AUGMENTED, VAL, CommSignal2),
    TEST:  get_unet_path(STAGE_AUGMENTED, TEST, CommSignal2)
}

In [ ]:
def load_scaling_factor():
    if SCALING_FACTORS_FILE.exists():
        with open(SCALING_FACTORS_FILE, 'r') as f:
            factor = float(f.read().strip())
        print(f"Loaded Global Scaling Factor: {factor:.6f}")
        return factor
    else:
        raise FileNotFoundError(f"Missing artifact: {SCALING_FACTORS_FILE}")

In [ ]:
def create_final_pairs(split, scaling_factor):
    print(f"\n--- Assembling Final Pairs for {split.upper()} ---")
    
    # Input X (Noisy) is already scaled in Stage 06
    path_x_scaled = get_unet_path(STAGE_SCALED, split, MIXED_DATASET)
    path_metadata = get_unet_path(STAGE_SCALED, split, MIXED_METADATA, "csv")
    path_y_source = CLEAN_SOURCES[split]
    
    # Destinations in Stage 07
    dest_x = get_unet_path(STAGE_FINAL, split, f"X_{split}")
    dest_y = get_unet_path(STAGE_FINAL, split, f"Y_{split}")
    

    # Validation Checks
    if not path_x_scaled.exists():
        print(f"Skipping {split}: No scaled input found at {path_x_scaled}")
        return
    if not path_y_source.exists():
        print(f"Skipping {split}: Clean source file missing at {path_y_source}")
        return

    # Process Input (X)
    # Since X is already scaled, we just copy it to the final folder for cleanliness
    print(f"  -> Copying X (Input) to {dest_x.name}...")
    shutil.copy(path_x_scaled, dest_x)

    # Process Target (Y)
    print(f"  -> Generating Y (Target) from {path_y_source.parent.name}...")
    
    # Load Metadata to find alignment
    df = pd.read_csv(path_metadata)
    
    clean_indices = df['clean_frame_id'].values
    clean_source_data = np.load(path_y_source, mmap_mode='r')
    
    if split == TRAIN:
        # CASE A: Training (Augmented)
        # The source file is ALREADY sliced to (N, 2, 40960). 
        # We just grab the rows.
        targets = clean_source_data[clean_indices]
    else:
        # CASE B: Val/Test (Raw)
        # The source file is (N, 2, 43560).
        # We must slice it to match X (40960).
        # Protocol: Deterministic Anchor -> Take first 40960 samples.
        print(f"     [Applying Deterministic Slice: 0 -> {FRAME_LENGTH}]")
        
        # Select the correct clean files
        raw_targets = clean_source_data[clean_indices]
        
        # Slice the time dimension (Axis 2 is time: [Batch, Channel, Time])
        # shape is (N, 43560, 2), change to [:, 0:FRAME_LENGTH, :]
        targets = raw_targets[:, :, 0:FRAME_LENGTH]

    
    # SCALE THE TARGETS
    # We divide by the SAME global factor so the model learns unity gain
    targets_scaled = targets / scaling_factor
    
    # Save Y
    np.save(dest_y, targets_scaled)
    print(f"  -> Saved Y (Target) to {dest_y.name}")
    print(f"   {split.upper()} Complete. Shapes -> X: {np.load(dest_x, mmap_mode='r').shape}, Y: {targets_scaled.shape}")

In [ ]:
global_max = load_scaling_factor()

for split in [TRAIN, VAL, TEST]:
    create_final_pairs(split, global_max)

print("\n---------------------------------------------------")
print("Final Dataset Ready for U-Net Training.")
print(f"Location: {UNET_PROCESSED_DATA_DIR / STAGE_FINAL}")